# 2.4 전이 동역학

전이 함수 T(s'|s,a)는 강화학습의 핵심이다. LLM 에이전트에서 온도(temperature) 파라미터는 결정적 vs 확률적 전이를 제어한다.

## 전이 함수와 동역학

### 전이 함수 T(s'|s,a)의 정의
상태 s에서 행동 a를 취했을 때, 다음 상태 s'로 전이될 **조건부 확률** (혹은 결정적 결과)이다.

- **결정적 전이**: T(s'|s,a) ∈ {0, 1} (항상 같은 결과)
- **확률적 전이**: T(s'|s,a) ∈ [0, 1] (여러 결과 가능, 확률로 표현)

### LLM의 온도(Temperature) 파라미터
- **temperature = 0**: 가장 확률이 높은 다음 토큰을 항상 선택 → 결정적
- **0 < temperature < 1**: 낮은 온도, 높은 토큰들에 집중 → 부분적 확률성
- **temperature = 1**: 기본값, 균형잡힌 분포
- **temperature > 1**: 높은 온도, 다양한 토큰 선택 가능 → 확률적

### 환경 모델로서의 LLM
LLM은 두 가지 역할을 한다:
1. **에이전트**: 상태를 관찰하고 행동 a를 선택
2. **환경 모델**: 다음 상태 s' = T(s, a)를 시뮬레이션

In [1]:
from utils_openai import *

## 실습 1: 결정적 vs 확률적 전이

In [2]:
heading("결정적 전이 (temperature=0)")

prompt = "2+2는 몇인가?"

# 결정적 전이: 항상 같은 응답
responses_deterministic = []
for i in range(3):
    resp = ask(prompt, temperature=0.0)  # T(s'|s,a) = 1.0 for one outcome
    responses_deterministic.append(resp)
    step_print(i+1, f"시도 {i+1}", resp[:60] + "...")

# 결정적인지 확인
all_same = all(r == responses_deterministic[0] for r in responses_deterministic)
step_print(4, "모두 동일한가?", "Yes" if all_same else "No")

kv_print([
    ("온도", "0.0"),
    ("전이 함수", "T(s'|s,a) = 1.0 (한 가지 결과만)"),
    ("결정적", "Yes")
])
print("\
✓ 결정적 전이에서는 항상 같은 상태로 전이된다.")


────────────────────────────────────────
  결정적 전이 (temperature=0)
────────────────────────────────────────
  [Step 1] 시도 1
    2 + 2는 4입니다....
  [Step 2] 시도 2
    2 + 2는 4입니다....
  [Step 3] 시도 3
    2 + 2는 4입니다....
  [Step 4] 모두 동일한가?
    Yes
  온도     0.0
  전이 함수  T(s'|s,a) = 1.0 (한 가지 결과만)
  결정적    Yes
✓ 결정적 전이에서는 항상 같은 상태로 전이된다.


## 실습 2: 확률적 전이

In [3]:
heading("확률적 전이 (temperature=1.0)")

prompt = "날씨가 아름답다는 문장을 다시 쓰시오."

# 확률적 전이: 다양한 응답
responses_stochastic = []
for i in range(4):
    resp = ask(prompt, temperature=1.0)  # T(s'|s,a) distributes over outcomes
    responses_stochastic.append(resp)
    step_print(i+1, f"시도 {i+1}", resp[:50] + "...")

# 다양성 확인
unique_responses = len(set(responses_stochastic))
step_print(5, "고유 응답 수", f"{unique_responses}/{len(responses_stochastic)}")

kv_print([
    ("온도", "1.0"),
    ("전이 함수", "T(s'|s,a) 분포 (여러 결과 가능)"),
    ("확률적", "Yes")
])
print("\
✓ 확률적 전이에서는 여러 가능한 상태로 전이될 수 있다.")


────────────────────────────────────────
  확률적 전이 (temperature=1.0)
────────────────────────────────────────
  [Step 1] 시도 1
    날씨가 화창하다....
  [Step 2] 시도 2
    날씨가 맑고 화창하다....
  [Step 3] 시도 3
    날씨가 화창하다....
  [Step 4] 시도 4
    날씨가 화창하다....
  [Step 5] 고유 응답 수
    2/4
  온도     1.0
  전이 함수  T(s'|s,a) 분포 (여러 결과 가능)
  확률적    Yes
✓ 확률적 전이에서는 여러 가능한 상태로 전이될 수 있다.


## World Model로서의 LLM

### 환경 모델링
LLM이 환경의 전이 함수를 학습했다고 생각할 수 있다:
- LLM은 프롬프트(상태)를 입력받아
- 그 다음 가능한 텍스트 토큰들의 확률 분포를 생성
- 이는 환경이 어떻게 반응할지를 모델링하는 것

### 근사 환경 모델
```
s' = T(s, a) 근사 = LLM(s, a, temperature)
```

### 한계
1. LLM은 도구 실행 결과를 모를 수 없다
2. 실시간 외부 환경 변화를 모델링할 수 없다
3. 학습 데이터의 cutoff로 인해 신규 정보를 모를 수 있다

따라서 실제 환경과의 상호작용이 필요하다.

## 실습 3: LLM 기반 환경 시뮬레이션

In [4]:
heading("환경 모델로서의 LLM")

# 현재 상태: 에이전트의 행동
current_action = "안녕하세요. 저는 파이썬에 관해 배우고 싶습니다."
step_print(1, "에이전트 행동", current_action)

# LLM이 다음 상태를 예측 (환경 모델)
environment_prompt = f"대화 상황: 사용자가 '{current_action}'라고 했다. 당신은 도우미이다. 어떻게 응답하겠는가?"

# 여러 온도로 다음 상태 샘플링
next_states = {}
for temp in [0.0, 0.5, 1.0]:
    next_state = ask(environment_prompt, temperature=temp)
    next_states[temp] = next_state
    step_print(2, f"환경 응답 (T={temp})", next_state[:50] + "...")

# 온도가 높을수록 다양한 다음 상태가 생성됨
kv_print([
    ("현재 상태", current_action[:40] + "..."),
    ("전이 함수", "LLM으로 근사"),
    ("다음 상태 수", "온도에 따라 1개~다수")
])
print("\
✓ LLM이 환경의 다음 상태를 시뮬레이션할 수 있다.")


────────────────────────────────────────
  환경 모델로서의 LLM
────────────────────────────────────────
  [Step 1] 에이전트 행동
    안녕하세요. 저는 파이썬에 관해 배우고 싶습니다.
  [Step 2] 환경 응답 (T=0.0)
    안녕하세요! 파이썬에 관심을 가져주셔서 감사합니다. 파이썬은 배우기 쉽고 다양한 용도로 사...
  [Step 2] 환경 응답 (T=0.5)
    안녕하세요! 파이썬에 관심이 있으시다니 정말 좋네요. 파이썬은 배우기 쉽고 다양한 분야에서...
  [Step 2] 환경 응답 (T=1.0)
    안녕하세요! 파이썬에 관심이 있으시다니 좋네요. 파이썬은 배우기 쉽고 다양한 분야에서 사용...
  현재 상태    안녕하세요. 저는 파이썬에 관해 배우고 싶습니다....
  전이 함수    LLM으로 근사
  다음 상태 수  온도에 따라 1개~다수
✓ LLM이 환경의 다음 상태를 시뮬레이션할 수 있다.


## 이론: 다단계 전이와 경로 의존성

### 경로 의존성 (Path Dependency)
마르코프 성질에 따르면 다음 상태는 현재 상태에만 의존해야 한다.
하지만 실제로는 이전의 전체 경로에 영향을 받을 수 있다:

```
경로 1: s0 →a1→ s1 →a2→ s2  (결과: R=10)
경로 2: s0 →a1'→ s1' →a2→ s2' (결과: R=5)
```

같은 최종 상태라도 도달 경로에 따라 보상이 다를 수 있다.

### 다단계 대화에서의 경로 의존성
1. "당신의 이름은?" → "저는 도우미입니다"
2. "당신의 이름은?" → "저는 Claude입니다"

같은 질문이지만, 이전 대화에 따라 다른 응답이 나온다.

In [7]:
heading("다단계 전이 추적")

memory = MemoryStream()
trajectory = []

# 초기 상태
s0 = "사용자: 안녕하세요"
memory.add(s0)
trajectory.append(s0)
step_print(1, "s0 (초기 상태)", s0)

# 첫 번째 전이: a1 적용
a1 = "도우미가 친절하게 인사한다"
s1 = ask(f"'{s0}'에 대해 친절하게 응답하시오.", temperature=0.5)
memory.add(f"응답: {s1}")
trajectory.append(s1)
step_print(2, "a1 (첫 행동)", a1)
step_print(3, "s1 (전이 후)", s1[:50] + "...")

# 두 번째 전이: a2 적용
a2 = "사용자의 질문에 자세히 답한다"
follow_up = "사용자: 당신은 무엇을 도와주나요?"
memory.add(follow_up)
s2 = ask(f"이전 응답: {s1}\
사용자: {follow_up}\
자세히 답하시오.", temperature=0.5)
memory.add(f"응답: {s2}")
trajectory.append(s2)
step_print(4, "a2 (두 번째 행동)", a2)
step_print(5, "s2 (전이 후)", s2[:50] + "...")

# 경로 확인
kv_print([
    ("경로", "s0 →a1→ s1 →a2→ s2"),
    ("단계 수", "2 (2개 전이)"),
    ("경로 의존성", "있음 (s2는 s0, s1, a2 모두에 의존)")
])
print("\
✓ 다단계 대화에서 경로 의존성이 중요하다.")


────────────────────────────────────────
  다단계 전이 추적
────────────────────────────────────────
  [Step 1] s0 (초기 상태)
    사용자: 안녕하세요
  [Step 2] a1 (첫 행동)
    도우미가 친절하게 인사한다
  [Step 3] s1 (전이 후)
    안녕하세요! 어떻게 도와드릴까요? 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요....
  [Step 4] a2 (두 번째 행동)
    사용자의 질문에 자세히 답한다
  [Step 5] s2 (전이 후)
    안녕하세요! 저는 다양한 주제에 대해 정보를 제공하고 질문에 답변하는 데 도움을 드릴 수 ...
  경로      s0 →a1→ s1 →a2→ s2
  단계 수    2 (2개 전이)
  경로 의존성  있음 (s2는 s0, s1, a2 모두에 의존)
✓ 다단계 대화에서 경로 의존성이 중요하다.


## 실습 4: 대화의 길이와 전이의 정확성

In [9]:
heading("긴 대화에서의 전이 동역학")

memory = MemoryStream()
conversation_length = 0

# 긴 대화 시뮬레이션
interactions = [
    ("사용자: 안녕하세요", "인사"),
    ("사용자: 날씨는 어떻게 되나요?", "날씨 질문"),
    ("사용자: 내일은 어떨까요?", "미래 날씨 질문"),
    ("사용자: 우산이 필요할까요?", "도구 필요성 질문"),
    ("사용자: 감사합니다.", "감사의 인사")
]

for i, (interaction, label) in enumerate(interactions):
    memory.add(interaction)
    conversation_length += len(interaction)
    step_print(i+1, f"Turn {i+1}: {label}", interaction[:40] + "...")

# 현재 상태 크기
full_history = "\n".join([t[0] for t in interactions])
state_size = len(full_history)

# 다음 상태 예측
next_prompt = f"다음 대화:\
{full_history}\
사용자: 당신의 이름은?\
응답:"
next_response = ask(next_prompt, temperature=0.5, max_tokens=100)

step_print(len(interactions)+1, "다음 상태 예측", next_response[:50] + "...")

kv_print([
    ("대화 길이", f"{len(interactions)} 턴"),
    ("상태 크기", f"{state_size} 글자"),
    ("전이 함수 입력", f"{len(next_prompt)} 글자"),
    ("장점", "전체 맥락 활용"),
    ("단점", "긴 입력으로 인한 비용")
])
print("\
✓ 긴 대화에서는 전체 상태가 더 큰 입력으로 LLM에 전달된다.")


────────────────────────────────────────
  긴 대화에서의 전이 동역학
────────────────────────────────────────
  [Step 1] Turn 1: 인사
    사용자: 안녕하세요...
  [Step 2] Turn 2: 날씨 질문
    사용자: 날씨는 어떻게 되나요?...
  [Step 3] Turn 3: 미래 날씨 질문
    사용자: 내일은 어떨까요?...
  [Step 4] Turn 4: 도구 필요성 질문
    사용자: 우산이 필요할까요?...
  [Step 5] Turn 5: 감사의 인사
    사용자: 감사합니다....
  [Step 6] 다음 상태 예측
    안녕하세요! 저는 AI 어시스턴트입니다. 당신의 질문에 답변해드리기 위해 여기 있습니다. ...
  대화 길이     5 턴
  상태 크기     71 글자
  전이 함수 입력  93 글자
  장점        전체 맥락 활용
  단점        긴 입력으로 인한 비용
✓ 긴 대화에서는 전체 상태가 더 큰 입력으로 LLM에 전달된다.


## 핵심 정리

| 개념 | 정의 | 효과 |
|------|------|------|
| **전이 함수 T(s'\|s,a)** | 상태와 행동으로부터 다음 상태 결정 | 에이전트 행동의 결과 |
| **온도 (Temperature)** | 0~∞ 값으로 생성 다양성 제어 | T=0: 결정적, T>1: 확률적 |
| **환경 모델** | LLM이 다음 상태를 예측 | World Model 역할 |
| **경로 의존성** | 이전 상태들이 현재 전이에 영향 | 대화 히스토리의 중요성 |
| **상태 크기** | 길어질수록 증가 | 맥락 윈도우 제한 문제 |

### 온도별 전이 특성
| 온도 | 범위 | 결과의 다양성 | 사용 시기 |
|------|------|------------|--------|
| **0.0** | 결정적 | 낮음 | 정확성이 중요한 작업 |
| **0.3-0.7** | 낮음 | 중간 | 일반적인 대화 |
| **1.0** | 중간 | 높음 | 창의성이 필요한 작업 |
| **>1.0** | 높음 | 매우 높음 | 다양한 결과 필요 |